# 04. Application + предыдущие заявки

## Цель

Кратко:
- создать признаки из `previous_application.csv`;
- добавить только эти признаки к `application_train`;
- обучить CatBoost и записать `application_previous_application`.


## 1. Импорты и пути


In [1]:
from pathlib import Path
import sys


def _is_project_root(path):
    return (
        (path / "src").is_dir()
        and (path / "notebooks").is_dir()
        and (path / "data").is_dir()
    )


project_candidates = [
    Path.cwd(),
    *Path.cwd().parents,
    Path("/content/credit-scoring-system"),
]

if "google.colab" in sys.modules:
    from google.colab import drive

    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.is_dir():
        drive.mount("/content/drive")

    default_drive_project = (
        drive_root / "credit-scoring-system"
    )
    project_candidates.append(default_drive_project)

    if not any(
        _is_project_root(path)
        for path in project_candidates
    ):
        project_candidates.extend(
            config_path.parents[1]
            for config_path in drive_root.rglob("src/config.py")
        )

PROJECT_ROOT = next(
    (
        path.resolve()
        for path in project_candidates
        if _is_project_root(path)
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не найден корень credit-scoring-system. На Google Drive "
        "должна находиться вся папка проекта с src/, notebooks/ и data/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_setup import setup_notebook


PROJECT_ROOT = setup_notebook()

Mounted at /content/drive
Installing missing dependency: catboost
Environment: Google Colab
Python: 3.12.13
Project root: /content/drive/MyDrive/credit-scoring-system
Raw data: /content/drive/MyDrive/credit-scoring-system/data/raw
Models: /content/drive/MyDrive/credit-scoring-system/models
Reports: /content/drive/MyDrive/credit-scoring-system/reports


In [2]:
import numpy as np
import pandas as pd
from catboost import (
    CatBoostClassifier,
    Pool,
    cv as catboost_cv,
)
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

from src.config import (
    INTERIM_DATA_DIR as DATA_INTERIM_DIR,
    PROCESSED_DATA_DIR as DATA_PROCESSED_DIR,
    find_data_file,
)
from src.experiment_tracking import save_experiment_result
from src.model_config import (
    get_catboost_device_config,
    get_catboost_gpu_count,
    print_catboost_device_info,
)


APPLICATION_PATH = find_data_file("application_train.csv")
PREVIOUS_PATH = find_data_file("previous_application.csv")
CLIENT_SPLIT_PATH = (
    DATA_PROCESSED_DIR / "client_split.csv"
)
PREVIOUS_FEATURES_PATH = (
    DATA_INTERIM_DIR / "previous_application_features.csv"
)
RANDOM_STATE = 42
CV_FOLDS = 3


### Устройство CatBoost


In [3]:
gpu_count = get_catboost_gpu_count()
catboost_device_config = get_catboost_device_config(
    gpu_count=gpu_count,
)
print_catboost_device_info(
    catboost_device_config,
    gpu_count=gpu_count,
)


Modeling environment: Google Colab
CatBoost GPU count: 1
CatBoost device: GPU
CatBoost GPU devices: 0


## 2. Загрузка application_train

Основная таблица содержит одну строку на клиента. Техническое значение
`365243` в `DAYS_EMPLOYED` заменяется пропуском.


In [4]:
application = pd.read_csv(APPLICATION_PATH)

if "DAYS_EMPLOYED" in application.columns:
    application["DAYS_EMPLOYED"] = application[
        "DAYS_EMPLOYED"
    ].replace(365243, np.nan)

assert application["SK_ID_CURR"].is_unique
assert application["TARGET"].isin([0, 1]).all()

print("Application:", application.shape)
display(application.head())


Application: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Загрузка previous_application


In [5]:
previous = pd.read_csv(PREVIOUS_PATH)

assert {"SK_ID_CURR", "SK_ID_PREV"}.issubset(previous.columns)

print("Previous application:", previous.shape)
display(previous.head())


Previous application: (1670214, 37)


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Признаки на уровне предыдущей заявки

Используются только решения не позднее текущей заявки. Флаги решений и
отношение выданной суммы к запрошенной рассчитываются до агрегации.


In [6]:
previous = previous[
    previous["DAYS_DECISION"].le(0)
].copy()

date_columns = [
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION",
]

previous[date_columns] = previous[
    date_columns
].replace(365243, np.nan)

previous["IS_APPROVED"] = (
    previous["NAME_CONTRACT_STATUS"].eq("Approved")
).astype(int)

previous["IS_REFUSED"] = (
    previous["NAME_CONTRACT_STATUS"].eq("Refused")
).astype(int)

previous["CREDIT_TO_APPLICATION_RATIO"] = (
    previous["AMT_CREDIT"]
    / previous["AMT_APPLICATION"].replace(0, np.nan)
)


## 5. Агрегация до клиента


In [7]:
previous_features = (
    previous
    .groupby("SK_ID_CURR")
    .agg(
        PREV_APPLICATION_COUNT=("SK_ID_PREV", "count"),
        PREV_APPROVED_SHARE=("IS_APPROVED", "mean"),
        PREV_REFUSED_SHARE=("IS_REFUSED", "mean"),
        PREV_REQUESTED_AMOUNT_MEAN=("AMT_APPLICATION", "mean"),
        PREV_REQUESTED_AMOUNT_TOTAL=("AMT_APPLICATION", "sum"),
        PREV_GRANTED_AMOUNT_MEAN=("AMT_CREDIT", "mean"),
        PREV_GRANTED_AMOUNT_TOTAL=("AMT_CREDIT", "sum"),
        PREV_CREDIT_TO_APPLICATION_MEAN=(
            "CREDIT_TO_APPLICATION_RATIO",
            "mean",
        ),
        PREV_DOWN_PAYMENT_MEAN=("AMT_DOWN_PAYMENT", "mean"),
        PREV_RATE_DOWN_PAYMENT_MEAN=("RATE_DOWN_PAYMENT", "mean"),
        PREV_MOST_RECENT_DECISION_DAYS=("DAYS_DECISION", "max"),
    )
    .reset_index()
)

previous_features[
    "PREV_MOST_RECENT_DECISION_DAYS"
] *= -1

assert previous_features["SK_ID_CURR"].is_unique

print(previous_features.shape)
display(previous_features.head())


(338857, 12)


,SK_ID_CURR,PREV_APPLICATION_COUNT,PREV_APPROVED_SHARE,PREV_REFUSED_SHARE,PREV_REQUESTED_AMOUNT_MEAN,PREV_REQUESTED_AMOUNT_TOTAL,PREV_GRANTED_AMOUNT_MEAN,PREV_GRANTED_AMOUNT_TOTAL,PREV_CREDIT_TO_APPLICATION_MEAN,PREV_DOWN_PAYMENT_MEAN,PREV_RATE_DOWN_PAYMENT_MEAN,PREV_MOST_RECENT_DECISION_DAYS
0,100001,1,1.0,0.0,24835.50,24835.5,23787.00,23787.0,0.957782,2520.0,0.104326,1740
1,100002,1,1.0,0.0,179055.00,179055.0,179055.00,179055.0,1.000000,0.0,0.000000,606
2,100003,3,1.0,0.0,435436.50,1306309.5,484191.00,1452573.0,1.057664,3442.5,0.050030,746
3,100004,1,1.0,0.0,24282.00,24282.0,20106.00,20106.0,0.828021,4860.0,0.212008,815
4,100005,2,0.5,0.0,22308.75,44617.5,20076.75,40153.5,0.899950,4464.0,0.108964,315


## 6. Проверка и сохранение признаков


In [8]:
assert previous_features["SK_ID_CURR"].is_unique
assert "TARGET" not in previous_features.columns

previous_features.to_csv(
    PREVIOUS_FEATURES_PATH,
    index=False,
)

print("Сохранено:", PREVIOUS_FEATURES_PATH)


Сохранено: /content/drive/MyDrive/credit-scoring-system/data/interim/previous_application_features.csv


## 7. Merge с application


In [9]:
modeling_data = application.merge(
    previous_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)

assert len(modeling_data) == len(application)
assert modeling_data["SK_ID_CURR"].is_unique

print("Application:", application.shape)
print("После добавления previous_application:", modeling_data.shape)
print(
    "Добавлено признаков:",
    modeling_data.shape[1] - application.shape[1],
)


Application: (307511, 122)
После добавления previous_application: (307511, 133)
Добавлено признаков: 11


## Чтение единого client split


In [10]:
if not CLIENT_SPLIT_PATH.exists():
    raise FileNotFoundError(
        "Сначала выполните notebooks/02_application_baseline.ipynb. "
        f"Ожидаемый файл: {CLIENT_SPLIT_PATH}"
    )

client_split = pd.read_csv(CLIENT_SPLIT_PATH)

assert client_split.columns.tolist() == ["SK_ID_CURR", "split"]
assert client_split["SK_ID_CURR"].is_unique
assert set(client_split["split"]) == {"train", "holdout"}
assert set(client_split["SK_ID_CURR"]) == set(application["SK_ID_CURR"])

modeling_data = modeling_data.merge(
    client_split,
    on="SK_ID_CURR",
    how="inner",
    validate="one_to_one",
)

assert len(modeling_data) == len(application)
assert modeling_data["SK_ID_CURR"].is_unique

print(client_split["split"].value_counts())


split
train      246008
holdout     61503
Name: count, dtype: int64


## 9. Создание X и y

`TARGET`, идентификатор клиента и техническая колонка разделения не
передаются модели.


In [11]:
train_data = modeling_data[
    modeling_data["split"].eq("train")
].copy()

n_holdout = int(
    modeling_data["split"].eq("holdout").sum()
)

feature_columns = [
    column
    for column in modeling_data.columns
    if column not in {
        "TARGET",
        "SK_ID_CURR",
        "split",
    }
]

X_train = train_data[feature_columns]
y_train = train_data["TARGET"].astype(int)

assert "TARGET" not in X_train.columns
assert "SK_ID_CURR" not in X_train.columns
assert "split" not in X_train.columns

print("Train:", X_train.shape)
print("Holdout clients (не используется):", n_holdout)


Train: (246008, 131)
Holdout clients (не используется): 61503


## 10. Подготовка данных для CatBoost

Категориальные пропуски заменяются строкой. Числовые `NaN` остаются без
изменений: CatBoost обрабатывает их самостоятельно.


In [12]:
categorical_columns = (
    X_train
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

X_train_catboost = X_train.copy()
X_train_catboost[categorical_columns] = (
    X_train_catboost[categorical_columns]
    .fillna("Unknown")
    .astype(str)
)

print("Категориальных признаков:", len(categorical_columns))


Категориальных признаков: 16


## 11. Стратифицированная кросс-валидация


In [13]:
cv_splitter = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


## 12. Pool для CatBoost


In [14]:
train_pool = Pool(
    data=X_train_catboost,
    label=y_train,
    cat_features=categorical_columns,
)


## 13. Параметры CatBoost


In [15]:
catboost_params = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "custom_metric": ["PRAUC:type=Classic"],
    "auto_class_weights": "Balanced",
    "random_seed": RANDOM_STATE,
    "allow_writing_files": False,
    "verbose": False,
    **catboost_device_config,
}


## 14. Библиотечная CV CatBoost

OOF-предсказания не требуются, поэтому используется `catboost.cv()`
без ручного цикла по фолдам. Test в CV не участвует.


In [16]:
catboost_cv_results = catboost_cv(
    pool=train_pool,
    params=catboost_params,
    folds=cv_splitter,
    early_stopping_rounds=100,
    as_pandas=True,
    verbose=100,
)


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:877: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(
Default metric period is 5 because AUC, PRAUC is/are not implemented for GPU


Training on fold [0/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7015582	best: 0.7015582 (0)	total: 218ms	remaining: 3m 38s
100:	test: 0.7475494	best: 0.7475494 (100)	total: 10.5s	remaining: 1m 33s
200:	test: 0.7501325	best: 0.7501325 (200)	total: 19.7s	remaining: 1m 18s
300:	test: 0.7531271	best: 0.7531271 (299)	total: 28.4s	remaining: 1m 6s
400:	test: 0.7548096	best: 0.7548096 (398)	total: 38.3s	remaining: 57.2s
500:	test: 0.7558984	best: 0.7558984 (500)	total: 47.6s	remaining: 47.4s
600:	test: 0.7566624	best: 0.7566872 (593)	total: 56.8s	remaining: 37.7s
700:	test: 0.7570784	best: 0.7570801 (697)	total: 1m 6s	remaining: 28.4s
800:	test: 0.7574905	best: 0.7575005 (799)	total: 1m 14s	remaining: 18.6s
900:	test: 0.7576595	best: 0.7577055 (866)	total: 1m 24s	remaining: 9.29s
999:	test: 0.7580690	best: 0.7580690 (999)	total: 1m 34s	remaining: 0us
bestTest = 0.7580689788
bestIteration = 999
Training on fold [1/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7050940	best: 0.7050940 (0)	total: 166ms	remaining: 2m 46s
100:	test: 0.7499655	best: 0.7499655 (99)	total: 10.1s	remaining: 1m 30s
200:	test: 0.7539459	best: 0.7539462 (198)	total: 18.7s	remaining: 1m 14s
300:	test: 0.7569112	best: 0.7569112 (299)	total: 28.5s	remaining: 1m 6s
400:	test: 0.7580680	best: 0.7580680 (394)	total: 37.3s	remaining: 55.6s
500:	test: 0.7586066	best: 0.7586066 (495)	total: 46.3s	remaining: 46.1s
600:	test: 0.7592278	best: 0.7592278 (597)	total: 55.9s	remaining: 37.1s
700:	test: 0.7597426	best: 0.7597426 (699)	total: 1m 4s	remaining: 27.3s
800:	test: 0.7606443	best: 0.7606553 (798)	total: 1m 13s	remaining: 18.3s
900:	test: 0.7616265	best: 0.7616265 (900)	total: 1m 23s	remaining: 9.21s
999:	test: 0.7620116	best: 0.7620116 (999)	total: 1m 32s	remaining: 0us
bestTest = 0.7620115876
bestIteration = 999
Training on fold [2/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7060657	best: 0.7060657 (0)	total: 230ms	remaining: 3m 49s
100:	test: 0.7501053	best: 0.7501053 (100)	total: 9.43s	remaining: 1m 23s
200:	test: 0.7526851	best: 0.7526901 (197)	total: 19.2s	remaining: 1m 16s
300:	test: 0.7535744	best: 0.7535744 (300)	total: 27.7s	remaining: 1m 4s
400:	test: 0.7555426	best: 0.7555426 (400)	total: 36.9s	remaining: 55.1s
500:	test: 0.7572633	best: 0.7573113 (490)	total: 46.8s	remaining: 46.6s
600:	test: 0.7584428	best: 0.7584428 (600)	total: 55s	remaining: 36.5s
700:	test: 0.7591902	best: 0.7591902 (700)	total: 1m 4s	remaining: 27.7s
800:	test: 0.7599390	best: 0.7599513 (794)	total: 1m 14s	remaining: 18.6s
900:	test: 0.7603580	best: 0.7603641 (896)	total: 1m 23s	remaining: 9.13s
999:	test: 0.7607764	best: 0.7607950 (992)	total: 1m 33s	remaining: 0us
bestTest = 0.7607949972
bestIteration = 992


## 15. Лучшая итерация и CV-метрики


In [17]:
auc_mean_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-mean")
)

auc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-std")
)

pr_auc_mean_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-mean")
)

pr_auc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-std")
)

best_cv_index = catboost_cv_results[
    auc_mean_column
].idxmax()

best_cv_row = catboost_cv_results.loc[
    best_cv_index
]

best_iteration = int(
    best_cv_row["iterations"]
) + 1

cv_roc_auc = float(
    best_cv_row[auc_mean_column]
)

cv_roc_auc_std = float(
    best_cv_row[auc_std_column]
)

cv_pr_auc = float(
    best_cv_row[pr_auc_mean_column]
)

cv_pr_auc_std = float(
    best_cv_row[pr_auc_std_column]
)

print(f"Лучшая итерация: {best_iteration}")
print(
    f"CV ROC-AUC: {cv_roc_auc:.4f} "
    f"± {cv_roc_auc_std:.4f}"
)
print(
    f"CV PR-AUC: {cv_pr_auc:.4f} "
    f"± {cv_pr_auc_std:.4f}"
)


Лучшая итерация: 1000
CV ROC-AUC: 0.7603 ± 0.0020
CV PR-AUC: nan ± nan


## 16. Итоговая модель CatBoost


In [18]:
catboost_model = CatBoostClassifier(
    iterations=best_iteration,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    verbose=100,
    **catboost_device_config,
)


## 17. Обучение итоговой модели


In [19]:
catboost_model.fit(
    X_train_catboost,
    y_train,
    cat_features=categorical_columns,
)


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 63.6ms	remaining: 1m 3s
100:	total: 5.8s	remaining: 51.6s
200:	total: 10.1s	remaining: 40.1s
300:	total: 14.4s	remaining: 33.4s
400:	total: 20.1s	remaining: 30.1s
500:	total: 24.4s	remaining: 24.3s
600:	total: 28.7s	remaining: 19s
700:	total: 34.2s	remaining: 14.6s
800:	total: 38.4s	remaining: 9.54s
900:	total: 42.6s	remaining: 4.68s
999:	total: 48.6s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', depth=6, devices='0', eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

## Запись CV-результата


In [20]:
current_result = {
    "experiment": "application_previous",
    "notebook": "04_previous_application_features.ipynb",
    "model": "CatBoostClassifier",
    "feature_set": "application + previous_application",
    "source_tables": "application_train.csv, previous_application.csv",
    "device": catboost_device_config["task_type"],
    "n_train": len(train_data),
    "n_holdout": n_holdout,
    "n_features": X_train.shape[1],
    "cv_folds": CV_FOLDS,
    "best_iteration": best_iteration,
    "cv_roc_auc": cv_roc_auc,
    "cv_roc_auc_std": cv_roc_auc_std,
    "cv_pr_auc": cv_pr_auc,
    "cv_pr_auc_std": cv_pr_auc_std,
    "holdout_roc_auc": None,
    "holdout_pr_auc": None,
}

all_results = save_experiment_result(current_result)
display(all_results)


,experiment,notebook,model,feature_set,source_tables,device,n_train,n_holdout,n_features,cv_folds,best_iteration,cv_roc_auc,cv_roc_auc_std,cv_pr_auc,cv_pr_auc_std,holdout_roc_auc,holdout_pr_auc
0,application_logistic,02_application_baseline.ipynb,LogisticRegression,application,application_train.csv,CPU,246008,61503,120,3,NaN,0.744841,0.002434,0.217865,0.005206,NaN,NaN
1,application_catboost,02_application_baseline.ipynb,CatBoostClassifier,application,application_train.csv,GPU,246008,61503,120,3,1000.0,0.754176,0.002317,NaN,NaN,NaN,NaN
2,application_bureau,03_bureau_features.ipynb,CatBoostClassifier,application + bureau,"application_train.csv, bureau.csv, bureau_bala...",GPU,246008,61503,134,3,1000.0,0.758362,0.001620,NaN,NaN,NaN,NaN
3,application_previous,04_previous_application_features.ipynb,CatBoostClassifier,application + previous_application,"application_train.csv, previous_application.csv",GPU,246008,61503,131,3,1000.0,0.760286,0.002017,NaN,NaN,NaN,NaN


## Выводы


In [21]:
print("Эксперимент: application + previous_application")
print(f"Количество признаков: {X_train.shape[1]}")
print(f"Лучшая итерация: {best_iteration}")
print(f"CV ROC-AUC: {cv_roc_auc:.4f}")
print(f"CV PR-AUC: {cv_pr_auc:.4f}")
print("Holdout не использовался: результат сравнивается только по CV.")


Эксперимент: application + previous_application
Количество признаков: 131
Лучшая итерация: 1000
CV ROC-AUC: 0.7603
CV PR-AUC: nan
Holdout не использовался: результат сравнивается только по CV.
